# L13b: Synthetic Markets of Many Stocks
Previously, we built a jump hidden Markov model of the daily growth rate of one asset. How do we simulate many stocks that move together, as real stocks in the same market and sector do?

> __Learning Objectives:__
>
> By the end of this lecture, you should be able to:
>
> * __Test the single-asset model out of sample:__ Compare paths simulated from the jump hidden Markov model of `SPY` with the daily growth rates that came after its fitting period. Turn the jump episodes off and compare both models, including their volatility clustering, on the same data.
> * __Compose a synthetic market of many stocks:__ Combine each stock's baseline with market, sector, and stock-specific residual components to build synthetic stocks that move together like real ones. Describe the jump hidden Markov models that supply the market, sector, and stock-specific draws.
> * __Size the stock-specific draw and sample the universe:__ Choose each stock's residual scale so that its sector and stock-specific draws keep the variance left after the market fit. Draw one synthetic path of every stock, holding each stock's market sensitivity fixed or letting it drift.

Some results in this lecture are core, and some are here to build understanding:

> __What you need to know:__
>
> __Core concepts:__
>
> * An out-of-sample test compares a fitted model with data from after its fitting period. We test the jump hidden Markov model of `SPY`, with and without jump episodes, on the growth rates of 2025 and 2026.
> * We model a synthetic stock's growth rate as $g_{i} = \alpha_{i} + \beta_{i}\,g_{m} + \delta_{i}\,f_{s} + \psi_{i}\,\tilde{u}_{i}$, with a stock-specific baseline $\alpha_{i}$. Shared market and sector components capture correlations between stocks. The stock-specific draw $\tilde{u}_{i}$ is centered on zero and represents the movement left after subtracting the baseline, market, and sector contributions.
> * The residual scale $\psi_{i}$ targets the stock's volatility. A drifting $\beta_{i}$ lets the stock's market sensitivity change over time.
>
> __For understanding__ (you will not need to reproduce or memorize these):
>
> * The tests that compare the `SPY` model with the out-of-sample data.
> * How the sector loadings and the residual scales are chosen, and the drifting $\beta$ option.
> * The sampling algorithm of the sector factor model, step by step.

__Notation:__ The [course notation page](https://varnerlab.org/CHEME-5660-CourseRepository-Fall-2026/dev/faq/notation.html#L13b) lists this lecture's symbols with their meanings and units.

Let's get started!

___


## Examples

__We will work through these examples in lecture:__

> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, let the sensitivities drift, and save the generator.

___


## Review: Synthetic Markets with Jump Hidden Markov Models
Previously, we built synthetic markets from jump hidden Markov models (JumpHMM). The model bins the daily growth rate into states of equal probability and estimates the transition matrix by counting moves between states.

Each simulated day, the chain takes one step. The model then draws the day's growth rate from the Student-t distribution of the new state:

<style>
  .course-diagram { color-scheme: light dark; }
  :host-context(body[data-vscode-theme-kind="vscode-light"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast-light"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-light"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast-light"] .course-diagram { color-scheme: light; }
  :host-context(body[data-vscode-theme-kind="vscode-dark"]) .course-diagram,
  :host-context(body[data-vscode-theme-kind="vscode-high-contrast"]) .course-diagram,
  body[data-vscode-theme-kind="vscode-dark"] .course-diagram,
  body[data-vscode-theme-kind="vscode-high-contrast"] .course-diagram { color-scheme: dark; }
  @media print { .course-diagram { color-scheme: only light !important; } }
</style>
<div>
    <center>
        <img class="course-diagram" src="figs/Fig-L13b-JumpHMM-Schematic.svg" width="880" style="max-width:100%; height:auto;" alt="A row of hidden states numbered 1, 2, through k, through N minus 1 and N, ordered from the most negative to the most positive daily growth rate, with double-headed arrows between neighbors. States 1 and 2 are red and braced as the most negative states S minus, and states N minus 1 and N are blue and braced as the most positive states S plus. A bell curve above each state is its emission, and state k emits mu k plus sigma k times a Student-t variable. Under state k, a note reads ordinary day: step with the estimated transition matrix. A dashed arrow from state k, marked no episode under way, leads up to a box: jump episode, starts with probability epsilon, lasts a Poisson number of days with mean lambda. From the box, a red arrow runs to the most negative states, taken on each episode day with probability p neg, and a blue arrow runs to the most positive states, taken with probability 1 minus p neg."/>
    </center>
</div>

Jump episodes start at random and last several days. On each episode day, the state comes from the most negative or the most positive states. These runs of large moves add the volatility clustering that the chain alone misses.

Let's test the jump hidden Markov model of `SPY` on data it has not seen:

> __Example:__
>
> [▶ Test the `SPY` model out of sample](CHEME-5660-L13b-Example-JumpHMM-SPY-OutOfSample-Fall-2026.ipynb). In this example, we load the jump hidden Markov model of `SPY` fitted to daily growth rates from 2014 to 2024. We compare its simulated paths with the growth rates of 2025 and 2026, which the model has not seen. We then turn the jump episodes off, score both models on the same data, and compare the volatility clustering of simulated and real growth rates.

__Today:__ We extend the jump hidden Markov model from one asset to many stocks that move together.

___


## Company Profile: AQR Capital Management
[AQR Capital Management](https://www.aqr.com/Our-Firm/About-Us) is a quantitative investment management firm. It was founded in New York City in 1998 and moved its headquarters to Greenwich, Connecticut, in 2004. The firm describes [its approach](https://www.aqr.com/What-We-Do/Our-Approach) as "systematic investing grounded in economic theory."

> __What does the firm do?__
>
> * __Betting against beta:__ In [Betting Against Beta](https://www.aqr.com/Insights/Research/Journal-Article/Betting-Against-Beta), published in 2014, AQR's Andrea Frazzini and Lasse H. Pedersen found that stocks with high beta tend to have low alpha. A portfolio that bought low-beta stocks with borrowed money and sold high-beta stocks short earned positive risk-adjusted returns.

__Explore further:__

* __Jobs and internships:__ AQR's [internships page](https://www.aqr.com/Our-Firm/Greenwich-Internships) describes a 10-week summer program in Greenwich. In November 2026, the [careers site](https://careers.aqr.com/) listed 2027 summer analyst roles in research, engineering, and business development.
* __Video:__ Watch [About AQR](https://www.youtube.com/embed/946GJrhyZtM) and [Internships at AQR](https://www.youtube.com/embed/Oow2UktGKcY) on the firm's YouTube channel.

__Connection to today's lecture:__ The betting-against-beta portfolio ranks stocks by their market sensitivity $\beta_{i}$, which sets the market component of our sector factor model. With a synthetic universe, we could test a betting-against-beta portfolio on many plausible market paths.

___


## Many Assets: The Sector Factor Model
We extend the single-asset jump hidden Markov model to many stocks with a factor model. The factor model splits each stock's daily growth rate into a fixed baseline plus market, sector, and stock-specific parts. We simulate each of these three parts with its own single-asset jump hidden Markov model.

Previously, we studied __factor models__, which describe the growth rate in terms of factors. For stock $i$ on day $t$, a general factor model is given by:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\sum_{j}\delta_{ij}\,f_{j}(t)}_{\text{other factors}} + \underbrace{\varepsilon_{i}(t)}_{\text{residual}}
$$

Here $\alpha_{i}$ is the stock-specific baseline (intercept), $g_{m}(t)$ is market growth, and $f_{j}(t)$ are other factors, such as interest rates or inflation. The coefficients $\beta_{i}$ and $\delta_{ij}$ measure sensitivity to these factors. The __residual__ $\varepsilon_{i}(t)$ is the growth left after subtracting the baseline and factor contributions.

The single index model keeps only the market factor. However, the single index residuals of stocks in the same sector, such as two banks, still move together. The sector factor model adds one factor for each sector of the Global Industry Classification Standard (GICS), such as Financials or Energy.

> __Definition (sector factor model):__ The daily growth rate of stock $i$ in sector $s$ has four parts:
>
> $$
> g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector}} + \underbrace{u_{i}(t)}_{\text{stock-specific}}
> $$
>
> Here $g_{m}(t)$ is the growth rate of `SPY`, $f_{s}(t)$ is the growth rate of sector $s$ with the market part removed, and $u_{i}(t)$ is the growth that neither the market nor the sector explains.

We compute the parts from the data in four steps: the market split, the sector factor, the sector loading, and the stock-specific residual.

### The Market Split
The single index fit of stock $i$ splits its growth rate into a baseline, a market part, and a single index residual:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market}} + \underbrace{\varepsilon_{i}(t)}_{\text{single index residual}}
$$

The market factor $g_{m}(t)$ is the growth rate of `SPY`. The single index residual $\varepsilon_{i}(t)$ is the growth that the market does not explain.

### The Sector Factor
The sector factor $f_{s}(t)$ is the growth rate of sector $s$ with the market part removed. We compute it as the average single index residual of the $n_{s}$ stocks in the sector.

We substitute the market split of each stock into this average:

$$
\begin{align*}
f_{s}(t) &= \frac{1}{n_{s}}\sum_{j\in s}\varepsilon_{j}(t) && \text{average single index residual}\\
&= \frac{1}{n_{s}}\sum_{j\in s}\left(g_{j}(t)-\alpha_{j}-\beta_{j}\,g_{m}(t)\right) && \text{market split of each stock } j\\
&= \frac{1}{n_{s}}\sum_{j\in s}g_{j}(t)-\frac{1}{n_{s}}\sum_{j\in s}\alpha_{j}-\left(\frac{1}{n_{s}}\sum_{j\in s}\beta_{j}\right)g_{m}(t) && \text{split the sum, and factor out } g_{m}(t)\\
&= \underbrace{\bar{g}_{s}(t)}_{\text{sector average}}-\underbrace{\bar{\alpha}_{s}}_{\text{baseline}}-\underbrace{\bar{\beta}_{s}\,g_{m}(t)}_{\text{market}} && \text{name the three averages}
\end{align*}
$$

Each bar is an average over the $n_{s}$ stocks in the sector: $\bar{g}_{s}(t)$ of their growth rates on day $t$, $\bar{\alpha}_{s}$ of their baselines, and $\bar{\beta}_{s}$ of their market sensitivities.

The $\beta_{i}\,g_{m}(t)$ term already carries the market part. If the sector factor kept the market part, a market move would enter the growth rate twice.

### Estimating the Sector Loading
Each stock in a sector moves with the sector factor by its own amount. The sector loading $\delta_{i}$ measures how strongly the single index residual $\varepsilon_{i}(t)=g_{i}(t)-\alpha_{i}-\beta_{i}\,g_{m}(t)$ of stock $i$ moves with its sector factor $f_{s}(t)$, as $\beta_{i}$ measures how strongly the stock moves with the market.

We estimate the sector loading $\delta_{i}$ by least squares, as we did for the $\alpha_{i}$ and $\beta_{i}$ parameters.

The regressor is the sector factor computed without stock $i$, the average single index residual of the other $n_{s}-1$ stocks in the sector:

$$
f_{s,-i}(t) = \frac{1}{n_{s}-1}\sum_{j\in s,\,j\neq i}\varepsilon_{j}(t).
$$

We leave stock $i$ out of this average, so its own single index residual does not appear on both sides of the fit. Both $\varepsilon_{i}(t)$ and $f_{s,-i}(t)$ average zero in the data, so the fit needs no intercept.

Stacking the fit for stock $i$ over the $T$ trading days of the data gives:

$$
\underbrace{\begin{bmatrix} \varepsilon_{i}(1)\\ \vdots\\ \varepsilon_{i}(T)\end{bmatrix}}_{\substack{\mathbf{y}\\ \text{single index residual of stock } i}}
=
\underbrace{\begin{bmatrix} f_{s,-i}(1)\\ \vdots\\ f_{s,-i}(T)\end{bmatrix}}_{\substack{\hat{\mathbf{X}}\\ \text{sector factor without stock } i}}
\,\hat{\delta}_{i}
+
\underbrace{\mathbf{e}}_{\text{fit errors}}
$$

where the vector $\mathbf{e}$ holds the error of the fit on each day. This is an overdetermined system, with $T$ equations and one unknown. No single value of $\delta$ makes every error zero.

Instead, the least-squares estimate minimizes the sum of squared errors, which has an analytical solution:

$$
\boxed{
\hat{\delta}_{i} = \arg\min_{\delta}\;\lVert\mathbf{y}-\hat{\mathbf{X}}\,\delta\rVert_{2}^{2}
\quad\Longrightarrow\quad
\left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)\hat{\delta}_{i} = \hat{\mathbf{X}}^{\top}\mathbf{y}
\quad\Longrightarrow\quad
\hat{\delta}_{i} = \left(\hat{\mathbf{X}}^{\top}\hat{\mathbf{X}}\right)^{-1}\hat{\mathbf{X}}^{\top}\mathbf{y}
}
$$

Because both series average zero, the solution is the sample covariance over the sample variance, like $\hat{\beta}_{i}$:

$$
\hat{\delta}_{i}
= \frac{\sum_{t=1}^{T}f_{s,-i}(t)\,\varepsilon_{i}(t)}{\sum_{t=1}^{T}f_{s,-i}(t)^{2}}
= \frac{\widehat{\operatorname{Cov}}\left(\varepsilon_{i},f_{s,-i}\right)}{\widehat{\operatorname{Var}}\left(f_{s,-i}\right)}.
$$

We keep only the slope $\hat{\delta}_{i}$ from this fit and discard the fit errors. We then multiply the slopes of each sector by one number $c_{s}$, so the sector loading is $\delta_{i} = c_{s}\,\hat{\delta}_{i}$. We choose $c_{s}$ so that the average covariance between the single index residuals of two stocks in the sector matches the data.

### The Stock-Specific Residual
With the sector loading $\delta_{i}$ in hand, we remove the sector part $\delta_{i}\,f_{s}(t)$ from the single index residual of stock $i$.

What is left is the stock-specific residual:

$$
u_{i}(t) = \underbrace{\varepsilon_{i}(t)}_{\text{single index residual}} - \underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector part}}
$$

The stock-specific residual $u_{i}(t)$ is the growth that neither the market nor the sector explains.

For example, if every bank rises on a day the market is flat, the move goes into the sector factor of Financials. If one bank rises alone after an earnings surprise, most of the move goes into the stock-specific residual of that bank.

Rearranging gives $\varepsilon_{i}(t) = \delta_{i}\,f_{s}(t) + u_{i}(t)$. Substituting this into the market split gives the four parts of the sector factor model.

### Simulating the Parts
Each part of the growth rate is a daily series in the data: the market factor $g_{m}(t)$, each sector factor $f_{s}(t)$, and each stock-specific residual $u_{i}(t)$. We fit each series with the same single-asset jump hidden Markov model that we fit to `SPY`, and tune its jump episodes the same way.

This gives three kinds of jump hidden Markov models:
* __Market model:__ For the market factor $g_{m}(t)$, we use the jump hidden Markov model of `SPY` that we tested out of sample. A path simulated from this model is a market draw.
* __Sector models:__ For each sector $s$, we fit a jump hidden Markov model to the sector factor $f_{s}(t)$ computed from the data. A path simulated from this model is a sector draw.
* __Stock-specific models:__ For each stock $i$, we fit a jump hidden Markov model to its stock-specific residual $u_{i}(t)$. A path simulated from this model is a stock-specific draw.

To simulate the universe, we draw one path from each model, independently of the other models. We then add the parts back together.

The simulated growth rate of stock $i$ in sector $s$ is:

$$
g_{i}(t) = \underbrace{\alpha_{i}}_{\text{baseline}} + \underbrace{\beta_{i}\,g_{m}(t)}_{\text{market draw}} + \underbrace{\delta_{i}\,f_{s}(t)}_{\text{sector draw}} + \underbrace{\psi_{i}\,\tilde{u}_{i}(t)}_{\text{stock-specific draw}}
$$

In this equation, $g_{m}(t)$ and $f_{s}(t)$ are the market and sector draws, not the data. Every stock shares the market draw, and every stock in a sector shares the sector draw. These shared draws are the only link between stocks.

__Centering the stock-specific draw:__ Let $u_{i}^{\mathrm{draw}}(t)$ be a draw from the stock-specific model of stock $i$, and $\bar{u}_{i}$ the model's long-run mean (the mean of one long simulated path). In the data, $u_{i}$ averages zero. We subtract the long-run mean, so the centered draw also averages zero:

$$
\tilde{u}_{i}(t) = u_{i}^{\mathrm{draw}}(t)-\bar{u}_{i}.
$$

__Sizing the stock-specific draw:__ The residual scale $\psi_{i}$ multiplies the centered draw. In the data, the sector factor averages the single index residuals of the stocks in the sector, stock $i$ included. In a sector of two stocks, half of the sector factor is the stock's own single index residual.

As a result, the stock-specific residual $u_{i}$ still moves with the sector factor in the data. The simulation draws the two independently. This drops the shared movement, and with it part of the stock's variance.

We choose $\psi_{i}$ so that the variances of the independent sector and stock-specific draws add up to the variance of the stock's single index residual in the data:

$$
\underbrace{\delta_{i}^{2}\operatorname{Var}(f_{s})}_{\text{sector variance}}
+ \underbrace{\psi_{i}^{2}\operatorname{Var}(u_{i})}_{\text{stock-specific variance}}
= \underbrace{\operatorname{Var}(\varepsilon_{i})}_{\text{variance left after the market fit}}.
$$

Solving for the scale gives:

$$
\boxed{\psi_{i}^{2} =
\frac{\operatorname{Var}(\varepsilon_{i})-\delta_{i}^{2}\operatorname{Var}(f_{s})}
{\operatorname{Var}(u_{i})}}
\qquad
\psi_{i} \geq 0.
$$

The variance equation sets a target. The fitted jump hidden Markov models reproduce the data variances only approximately. In a sector with many stocks, the stock's own share of the sector factor is small, and $\psi_{i}$ is close to one.

### Sampling Algorithm
The algorithm below draws one synthetic path of the whole universe over $T$ days.

__Initialize__: Given the fitted market model, sector models, and stock-specific models; each stock's baseline $\alpha_{i}$, market sensitivity $\beta_{i}$, sector loading $\delta_{i}$, and residual scale $\psi_{i}$; each stock's real starting price $S_{i}(0)$; the time step $\Delta t=1/252$ year (one trading day); and the number of days $T$.

1. __Shared draws:__ Draw one market path $g_{m}(1),\dotsc,g_{m}(T)$, shared by all stocks, from the market model. For each sector $s$ __do__: draw one factor path $f_{s}(1),\dotsc,f_{s}(T)$, shared by the stocks in that sector, from its sector model. Use independent random draws to generate the different paths.
2. __Stock-specific draws:__ For each stock $i$ __do__: draw a path from the stock-specific model of stock $i$ and subtract the model's long-run mean $\bar{u}_{i}$ to get $\tilde{u}_{i}(1),\dotsc,\tilde{u}_{i}(T)$.
3. __Compose:__ For each stock $i$ in sector $s$ and each day $t = 1,2,\dots,T$ __do__: compose the growth rate $g_{i}(t) \gets \alpha_{i} + \beta_{i}\,g_{m}(t) + \delta_{i}\,f_{s}(t) + \psi_{i}\,\tilde{u}_{i}(t)$, and compound the price, starting from $S_{i}(0)$: $S_{i}(t) \gets S_{i}(t-1)\exp\left(g_{i}(t)\,\Delta t\right)$.

__Output__: The market path $g_{m}(t)$, and the growth rates $g_{i}(t)$ and prices $S_{i}(t)$ of every synthetic stock, for $t = 1,\dotsc,T$.

__Note:__ With the drifting β option below, $\beta_{i}$ becomes $\beta_{i}(t)$ in step 3. We also reduce $\psi_{i}$ to preserve the variance target.

### An option: a drifting β
The coefficient $\beta_{i}$ measures stock $i$'s sensitivity to market growth. Holding $\beta_{i}$ fixed assumes that stock $i$'s sensitivity to market growth rate stays constant throughout a synthetic path. Alternatively, we can let the sensitivity change over time around its single index estimate $\hat{\beta}_{i}$. 

Each day, we update it with:

$$
\beta_{i}(t) = \hat{\beta}_{i}
+ \underbrace{(1-a)\left(\beta_{i}(t-1)-\hat{\beta}_{i}\right)}_{\text{retained deviation}}
+ \underbrace{a\,k_{i}\,z_{i}(t)}_{\text{new shock}}.
$$

Here $0<a<1$ is the exponential moving average weight. A small $a$ retains most of the previous day's deviation from $\hat{\beta}_{i}$.

The shocks $z_{i}(t)$ are independent standard normal draws, with their size controlled by $k_{i}\geq 0$. In the example, we choose $k_{i}$ using the variation in β estimates updated daily from real data. These estimates also contain sampling noise.

The market contribution is now $\beta_{i}(t)g_{m}(t)$. Let's look at an example:

> __Example:__
>
> [▶ Build a synthetic market of many stocks](CHEME-5660-L13b-Example-JumpHMM-SectorFactor-Fall-2026.ipynb). In this example, we build a universe of two stocks from each sector plus `SPY` with the sector factor model. We compare the correlations, volatilities, and market sensitivities of a synthetic path with the real data, let the sensitivities drift, and save the generator.

___

## Summary
We tested the jump hidden Markov model of `SPY` on data it had not seen. We then extended the model from one asset to a universe of stocks that move together, as real stocks in the same market and sector do.

> __Key Takeaways:__
>
> * __Testing the single-asset model out of sample:__ We compared paths simulated from the jump hidden Markov model of `SPY` with the growth rates of 2025 and 2026, which came after its fitting period. The paths with a jump episode failed the distribution tests more often, and their volatility clustering lasted longer than in the data.
> * __Shared draws make synthetic stocks move together:__ We combined each stock's baseline with shared market and sector components and a stock-specific residual component. Every stock shares the market draw, and every stock in a sector shares the sector draw.
> * __Variance targets and a drifting market sensitivity:__ We sized the stock-specific draw to preserve the variance left after the market fit when combined with the sector draw. As an option, we let each stock's market sensitivity drift around its single index estimate.

Synthetic markets of many stocks let us test a trading method on many plausible paths, rather than on one historical path. Knowing how the synthetic universe is built also tells us its limits: stocks in different sectors move together only through the market draw.

___


## Disclaimer and Risks
__This content is offered solely for training and informational purposes__. No offer or solicitation to buy or sell securities or derivative products or any investment or trading advice or strategy is made, given, or endorsed by the teaching team.

__Trading involves risk__. Carefully review your financial situation before investing in securities, futures contracts, options, or commodity interests. Past performance, whether actual or indicated by historical tests of strategies, is no guarantee of future performance or success.

Trading is generally inappropriate for someone with limited resources, investment or trading experience, or a low-risk tolerance. Only risk capital that is not required for living expenses should be used.

__You are fully responsible for any investment or trading decisions you make__. Such decisions should be based solely on evaluating your financial circumstances, investment or trading objectives, risk tolerance, and liquidity needs.

___
